In [ ]:
# 1. Import Library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Pratikum_Ml/pratikum05/data/stunting_wasting_dataset.csv")
df.head()

In [ ]:
df.info()

In [ ]:
#cek missing value
df.isnull().sum()

In [ ]:
#cek duplicate
df.duplicated().sum()

In [ ]:
#menghapus data duplikat
df = df.drop_duplicates()

In [ ]:
# cek setelah duplicate ulang setelah menghapus
df.duplicated().sum()

In [ ]:
df = df.rename(columns={
    'Jenis Kelamin': 'jenis_kelamin',
    'Umur (bulan)': 'umur_bulan',
    'Tinggi Badan (cm)': 'tinggi_cm',
    'Berat Badan (kg)': 'berat_kg'
})

In [ ]:
df.info()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='Stunting', data=df, palette='Set2')
plt.title('Distribusi Kategori Stunting')
plt.show()


In [ ]:
#mapping label -> kode untuk target
stunting_cat = df['Stunting'].astype('category')
stunting_classes = list(stunting_cat.cat.categories)#urutan kelas
df['Stunting'] = stunting_cat.cat.codes

#fitur kategorikal lain (jenis kelamin, wasting)->kode juga
for col in ['jenis_kelamin', 'wasting']:
  if col in df.columns:
    df[col] = df[col].astype('category').cat.codes

df.head()

In [ ]:
#KORELASI
# Mapping Wasting column to categorical codes
if 'Wasting' in df.columns:
  df['Wasting'] = df['Wasting'].astype('category').cat.codes

plt.figure (figsize=(6,4))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title('Korelasi Fitur')
plt.show()

In [ ]:
#memilih fitur dan target
feature_cols = ['umur_bulan', 'tinggi_cm', 'berat_kg', 'Wasting']
X = df[feature_cols]
y = df['Stunting']

In [ ]:
#membagi data set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)
len(X_train), len(X_test)

In [ ]:
#membangun model
dt = DecisionTreeClassifier(
    criterion= 'gini',
    max_depth=4,
    random_state=42
)
dt.fit(X_train, y_train)

In [ ]:
#evaluasi
y_pred = dt.predict (X_test)

print("Akurasi :", round(accuracy_score(y_test, y_pred)*100, 2), "%")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=[str(cls) for cls in stunting_classes]))

In [ ]:
plt.figure(figsize=(20,10))
plot_tree(
    dt,
    feature_names=X.columns,
    class_names=[str(c) for c in stunting_classes],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Visualisasi Decision Tree - Kasus Stunting")
plt.show()

In [ ]:
importance = pd.DataFrame({
    'Fitur': X.columns,
    'Importance': dt.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(6,4))
sns.barplot(x='Importance', y='Fitur', data=importance, palette='viridis')
plt.title('Feature Importance pada Model Decision Tree')
plt.show()

In [ ]:
# 13. Hyperparameter Tuning (Mencari max_depth terbaik)
# -----------------------------
scores = {}
for d in range(2, 10):
    m = DecisionTreeClassifier(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    acc = accuracy_score(y_test, m.predict(X_test))
    scores[d] = acc

best_depth = max(scores, key=scores.get)

print("\n===== Hasil Hyperparameter Tuning =====")
for d, s in scores.items():
    print(f"max_depth={d} -> Akurasi: {s:.4f}")
print(f"\nNilai max_depth terbaik: {best_depth} (Akurasi: {scores[best_depth]:.4f})")

plt.figure(figsize=(6,4))
plt.plot(list(scores.keys()), list(scores.values()), marker='o')
plt.title('Perbandingan Akurasi Berdasarkan max_depth')
plt.xlabel('max_depth')
plt.ylabel('Akurasi')
plt.grid(True)
plt.show()

#pratikum mandiri


In [ ]:
# 1. Import Library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Pratikum_Ml/pratikum05/data/Iris.csv")
df.head()

In [ ]:
# Menghapus kolom 'Id' karena tidak relevan untuk pemodelan
df = df.drop('Id', axis=1)

# Mengecek info data dan data yang hilang (missing values)
print("\nInfo Dataset (Setelah Hapus 'Id'):")
df.info()

print("\nCek Data Hilang (Missing Values):")
print(df.isnull().sum())
print("-" * 30)

# --- Encoding Fitur Target (Species) ---
# Model tidak mengerti teks ('Iris-setosa'), jadi kita ubah jadi angka (0, 1, 2)
# Ini adalah langkah yang sama seperti di praktikum stunting

# Menyimpan nama kelas aslinya untuk digunakan di laporan/visualisasi
species_names = df['Species'].unique()
print(f"Nama Kelas Unik: {species_names}")

# Menggunakan .astype('category').cat.codes untuk mengubah teks jadi angka
df['Species'] = df['Species'].astype('category').cat.codes

print("\nData Setelah Encoding (5 Baris Pertama):")
print(df.head())

In [ ]:
# 'X' adalah semua kolom fitur (semua kecuali 'Species')
X = df.drop('Species', axis=1)

# 'y' adalah kolom target
y = df['Species']

# Menyimpan nama-nama fitur untuk visualisasi pohon
feature_names = X.columns

print(f"\nFitur (X) yang digunakan: {list(feature_names)}")=
print(f"Target (y) adalah kolom 'Species'")

In [ ]:
# Membagi data sesuai instruksi tugas: 80% data latih, 20% data uji
# stratify=y memastikan proporsi kelas di data latih dan uji tetap seimbang
# random_state=42 memastikan hasil pembagian data konsisten
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)

print(f"\nJumlah data latih (80%): {len(X_train)} baris")
print(f"Jumlah data uji (20%): {len(X_test)} baris")

In [ ]:
# Membuat model Decision Tree
# Kita tentukan max_depth=5  dan random_state=42 untuk konsistensi
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=5, random_state=42)

# Melatih model menggunakan data latih (X_train dan y_train)
dt_model.fit(X_train, y_train)

print("\nModel Decision Tree berhasil dilatih.")

In [ ]:
# Menggunakan model yang sudah dilatih untuk memprediksi data uji (X_test)
y_pred = dt_model.predict(X_test)


# 1. Akurasi
acc = accuracy_score(y_test, y_pred)
print(f"\nAKURASI MODEL: {acc * 100:.2f}%")

# 2. Confusion Matrix
print("\nCONFUSION MATRIX:")
print(confusion_matrix(y_test, y_pred))

# 3. Classification Report
print("\nCLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred, target_names=species_names))

In [ ]:
# Membuat visualisasi pohon keputusan yang sudah dilatih
plt.figure(figsize=(20, 15))
plot_tree(
    dt_model,
    feature_names=feature_names,  # Nama fitur
    class_names=species_names,    # Nama kelas (target)
    filled=True,                  # Memberi warna pada node
    rounded=True,                 # Membuat sudut node lebih bulat
    fontsize=10
)
plt.title("Visualisasi Pohon Keputusan (Decision Tree) - Dataset Iris", fontsize=10)
plt.show()